<a href="https://colab.research.google.com/github/Nasim-Amani/ramp-aware-solar/blob/main/R_Q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ramp-aware LSTM forecasting of solar irradiance (10–60 s ahead)

## Aim
We forecast solar irradiance 10 to 60 seconds ahead with an LSTM, and study how well the forecasts capture **ramps**: sudden changes in irradiance caused by passing clouds. Standard models are trained and judged with MSE/RMSE, which rewards smooth forecasts. Such models can beat persistence on RMSE and still miss most ramps, which are the events that matter most for grid and battery control.

## What the data exploration showed
- Ramps occur almost only on broken-cloud days, are real (confirmed by an independent PV module) and are fast: most are completed within 10 s, and the sun often stays blocked for only about 20 s.
- On clear days persistence is almost perfect up to one minute; on broken-cloud days its error doubles within the first minute.
- From the past, the **chance** of a ramp is predictable, but its **exact timing and direction** hardly are. Ramps come in groups: once the sky becomes restless (many fast ups and downs from passing clouds), it usually stays restless for several minutes or longer, while calm periods also tend to stay calm. A model can therefore tell from the last few minutes whether a ramp is likely soon, but not exactly when the next cloud edge will arrive.

## Research questions
1. **Does a ramp-aware loss help?** Does adding a term on the size of the 10-s changes to the MSE loss improve ramp detection, and how much accuracy (RMSE skill) does it cost?
2. **Should the model predict the change instead of the value?** Does predicting the change from the current irradiance (and adding it to the current value) improve the forecasts, especially on clear days where persistence is almost perfect?
3. **Which inputs help?** Do time of day and measures of recent variability improve accuracy or ramp detection, compared with irradiance alone?
4. **What is the trade-off?** How do accuracy and ramp detection change as the weight λ of the ramp term increases?
5. **How robust are the results?** Do the conclusions hold for different lead times (10–60 s), for clear and broken-cloud days, and when each broken-cloud day is used as the test day?

## Setup
- **Data:** 1-s irradiance averaged to 10 s, daylight only, 11 complete days (16–26 Feb 2013).
- **Split by whole days, in time order:** train 16–23 Feb, validation 24 Feb, test 25 Feb (broken clouds) and 26 Feb (clear). Scaling uses the training days only.
- **Forecast:** input = the last 10 minutes; output = the next 6 values (10, 20, …, 60 s).
- **Model:** a single-layer LSTM (64 units), trained with early stopping. Each setup is trained with 3 random seeds and reported as mean ± standard deviation.

## Experiments

| | Question | What changes |
|---|---|---|
| **E1** | 1 and 2 | target (value vs change) × loss (MSE vs ramp-aware), irradiance only |
| **E2** | 3 | + time of day |
| **E3** | 3 | + recent-variability features |
| **E4** | 4 | weight λ of the ramp term (0, 0.5, 1, 3, 10) |
| **E5** | 5 | each broken-cloud day used as the test day in turn |

## Evaluation
- **Accuracy:** RMSE and skill against persistence, $\text{skill} = 1 - \text{RMSE}_{model} / \text{RMSE}_{persistence}$, for each lead time and separately for clear and broken-cloud days.
- **Ramps:** a ramp is a 10-s change larger than **80 W/m²** (95th percentile of the 10-s changes on the broken-cloud training days). We report precision, recall and F1 with a timing tolerance of ±10 s, on broken-cloud days. A threshold of 60 W/m² is used as a sensitivity check.

## Other features: what we have, when and how we include them

### 1. Other sensors

| Feature | What it measures | Expected use at 10–60 s | Why |
|---|---|---|---|
| `PV_2_GT_I` | current of PV module 2 | low | It sees the same clouds as the irradiance sensor (98.8 % same direction at ramp steps), so it adds almost no new information |
| `Amb_Tempr`, `Tempr_PV_1/2/3` | air and module temperatures | low | They change over minutes to hours, much slower than a cloud edge (about 10 s) |
| `PV_1_SA_V` | open-circuit voltage of module 1 | low | It mainly follows the module temperature, which changes slowly |
| `PV_4_*` | battery-charging panel | not usable | Its current and voltage are set by a charge controller, not by the sky |

### 2. Features computed from irradiance

| Feature | Expected use | Why |
|---|---|---|
| Recent variability (e.g. number of ramps in the last 5 min) | high | The chance of a ramp in the next minute rises from 3 % after a calm period to 55 % after many recent ramps |
| 1-s detail inside each 10-s step (min, max, std of the 1-s values) | medium | A cloud edge passes within seconds; averaging to 10 s hides where inside the step a change started |
| Clear-sky index instead of raw irradiance | medium | It removes the daily cycle of the sun, so the model sees only the effect of clouds |

### 3. Time and sun
Time of day is part of experiment E2. Sun elevation and clear-sky irradiance as additional raw inputs made the model worse in an earlier test (likely overfitting with only eight training days), so they are not added again.

### When
After the core experiments, in this order:
1. **E1:** target × loss, irradiance only (the base).
2. **E2:** + time of day.
3. **E3:** + recent variability.
4. **E3b:** + 1-s detail, then the clear-sky index.
5. **E3c:** + other sensors (temperatures, PV_1 voltage, PV_2 current), mainly to show whether they help at this time scale.

### How
1. **Quick screening (no training).** For each feature, we compute the mutual information with the future change of irradiance (ΔG over the next 60 s) on the training days. We use the future change and not the irradiance level, because almost every sensor follows the daily cycle of the sun and would look related to the level.
2. **Groups instead of single features.** Features are added in groups, each with its own physical reason (as in the tables above), and each setup is trained with 3 seeds.
3. **Keep only what helps.** A group is kept only if it improves validation skill or ramp F1 by more than the spread between seeds.
4. **Validation day only.** All decisions are made on the validation day; the test days are used once, at the end.

**Expectation:** features computed from irradiance (recent variability, 1-s detail) are the most promising; the other sensors probably add little at 10–60 s. Showing this with numbers is still a useful result.